In [2]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Perceptron

## Creating dataset

In [3]:
np.random.seed(42)
X = np.random.rand(1200).reshape(200, 6) > 0.5

In [4]:
y = []

for i in X:

    and_row = []

    if i[0] and i[1]:
        and_row.append(True)
    else:
        and_row.append(False)

    if [2] and i[3]:
        and_row.append(True)
    else:
        and_row.append(False)

    if [4] and i[5]:
        and_row.append(True)
    else:
        and_row.append(False)

    if sum(and_row) > 0:
        y.append(True)
    else:
        y.append(False)

In [5]:
y = np.array(y)

In [6]:
print(f"Доля положительных ответов - {sum(y) / len(y) * 100}")
print(f"Доля отрицательных ответов - {round((1 - sum(y) / len(y)) * 100, 2)}")

Доля положительных ответов - 83.5
Доля отрицательных ответов - 16.5


## Perceptron class

In [ ]:
class DummyPerceptron:

    def __init__(
            self, 
            learning_rate: float,
            seed: int
        ):
        self.seed          = seed
        self.learning_rate = learning_rate
        
        np.random.seed(self.seed)

    # Общие методы

    def _predict_row(
        self, 
        weights, 
        intercept, 
        x_row
    ):
        return 1 if (np.sum(weights * x_row) + intercept > 0.5) else 0

    def _compute_error(
        self,
        X,
        y,
        weights,
        intercept
    ):
        error = 0

        for i in range(X.shape[0]):
            predicted = self._predict_row(weights, intercept, X[i])
            error += (predicted - y[i]) ** 2

        return error

    def predict(self, X):
        return np.array([
            self._predict_row(self.weights, self.intercept_, X[i])
            for i in range(X.shape[0])
        ])

    def score(self, X, y):
        predictions = self.predict(X)
        return np.mean(predictions == y) * 100

    # Первый алгоритм

    def _weights_update(
            self, 
            X,
            y,
            row_index, 
            predicted_value
        ) -> None:
        delta = self.learning_rate * (predicted_value - y[row_index])
        self.weights    = self.weights - delta * X[row_index]
        self.intercept_ = self.intercept_ - delta * 1

    def learning_cycle(
            self, 
            X, 
            y
        ):

        for i in range(X.shape[0]):
            predicted_value = self._predict_row(self.weights, self.intercept_, X[i])
            self._weights_update(X, y, i, predicted_value)
            self.error += (predicted_value - y[i]) ** 2

    def fit(
            self, 
            X, 
            y, 
            max_epochs=1000,
            init_weights=None,
            init_intercept=None
        ):
        self.weights    = init_weights if init_weights is not None else np.random.rand(X.shape[1])
        self.intercept_ = init_intercept if init_intercept is not None else np.random.rand()

        for epoch in range(max_epochs):
            self.error = 0
            self.learning_cycle(X, y)

            if self.error == 0:
                print(f'Сошлось за {epoch + 1} эпох')
                break
        else:
            print('Не сошлось за max_epochs')

    # Метод Монте-Карло

    def fit_monte_carlo(self, X, y, N=1000):
        f_min                      = np.inf 
        weights_min, intercept_min = None, None 

        for i in range(N):
            weights_i   = np.random.rand(X.shape[1])
            intercept_i = np.random.rand()

            f_i = self._compute_error(X, y, weights_i, intercept_i)

            if f_i < f_min:
                f_min         = f_i 
                weights_min   = weights_i
                intercept_min = intercept_i

        self.weights    = weights_min
        self.intercept_ = intercept_min
        self.error      = f_min

    # Имитация отжига

    def _random_neighbor(
            self,
            weights,
            intercept,
            epsilon_step
    ):
        new_weights = weights + np.random.uniform(
            -epsilon_step,
            epsilon_step,
            size=weights.shape    
        )
        new_intercept = intercept + np.random.uniform(
            -epsilon_step,
            epsilon_step
        )
        return new_weights, new_intercept

    def fit_annealing(
            self,
            X,
            y,
            T_max=100.0,
            L=50,
            r=0.95,
            epsilon_step=0.1,
            eps=1e-10
    ):
        T = T_max 

        weights   = np.random.rand(X.shape[1])
        intercept = np.random.rand()
        f_x       = self._compute_error(X, y, weights, intercept)

        while T > eps:
            for _ in range(L):
                weights_new, intercept_new = self._random_neighbor(weights, intercept, epsilon_step)
                f_x_new = self._compute_error(X, y, weights_new, intercept_new)

                delta = f_x_new - f_x

                if delta <= 0:
                    weights, intercept, f_x = weights_new, intercept_new, f_x_new
                else:
                    if np.exp(-delta / T) > np.random.uniform(0, 1):
                        weights, intercept, f_x = weights_new, intercept_new, f_x_new

                if f_x == 0:
                    break 

            if f_x == 0:
                break 

            T = r * T 

        print(f'Отжиг завершен: ошибка = {f_x}, финальная T = {T}')

        self.fit(X, y, init_weights=weights, init_intercept=intercept)
        

## Split into train and test

In [78]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Using methods

In [88]:
for learn_rate in (0.01, 0.1, 1.0):

    print(f"\nМодель с learning rate: {learn_rate}")

    model = DummyPerceptron(learning_rate=learn_rate, seed=42)
    model.fit(X_train, y_train)

    print(f"Точность модели на обучаюищх данных {model.score(X_train, y_train)}")
    print(f"Точность модели на тестовых данных {model.score(X_test, y_test)}")
    print(f"Веса модели {model.weights}")


Модель с learning rate: 0.01
Сошлось за 16 эпох
Точность модели на обучаюищх данных 100.0
Точность модели на тестовых данных 100.0
Веса модели [0.29454012 0.39071431 0.18199394 0.71865848 0.02601864 0.61599452]

Модель с learning rate: 0.1
Сошлось за 4 эпох
Точность модели на обучаюищх данных 100.0
Точность модели на тестовых данных 100.0
Веса модели [ 0.37454012  0.35071431  0.03199394  0.79865848 -0.14398136  0.75599452]

Модель с learning rate: 1.0
Сошлось за 3 эпох
Точность модели на обучаюищх данных 100.0
Точность модели на тестовых данных 100.0
Веса модели [ 1.37454012  1.95071431 -0.26800606  3.59865848  0.15601864  4.15599452]


In [90]:
model_monte_carlo = DummyPerceptron(learning_rate=0.01, seed=42)
model_monte_carlo.fit_monte_carlo(X_train, y_train, N=10000)
print(f"Точность модели на обучаюищх данных {model_monte_carlo.score(X_train, y_train)}")
print(f"Точность модели на тестовых данных {model_monte_carlo.score(X_test, y_test)}")
print(f"Веса модели {model_monte_carlo.weights}")


Точность модели на обучаюищх данных 100.0
Точность модели на тестовых данных 100.0
Веса модели [0.302829   0.28827301 0.00770923 0.7426452  0.14531034 0.52064484]


In [91]:
model_temp = DummyPerceptron(learning_rate=0.01, seed=42)
model_temp.fit_annealing(X_train, y_train)
print(f"Точность модели на обучаюищх данных {model_temp.score(X_train, y_train)}")
print(f"Точность модели на тестовых данных {model_temp.score(X_test, y_test)}")
print(f"Веса модели {model_temp.weights}")

Отжиг завершен: ошибка = 0, финальная T = 2.3782688525533215
Сошлось за 1 эпох
Точность модели на обучаюищх данных 100.0
Точность модели на тестовых данных 97.5
Веса модели [ 0.55918175  0.31095194  0.06227928  1.20658638 -0.24446101  1.09807293]
